# RBL – LLM Test Generation & Branch Coverage Pipeline

Mỗi thành viên: chỉnh Bước 3 (chọn provider/model + secret key) rồi Runtime > Run all.
Chi tiết kiến trúc xem `README.md` ở gốc project.

## Bước 1 — Mount Google Drive dùng chung & vào thư mục project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/rbl_llm_testgen'
# Lần đầu: clone/copy project vào PROJECT_ROOT trước khi chạy cell này.
os.chdir(PROJECT_ROOT)
print('cwd:', os.getcwd())

## Bước 2 — Cài thư viện

In [ ]:
!pip install -q google-genai groq pyyaml coverage pytest pandas requests

## Bước 3 — Chọn provider/model + API key (CHỈNH Ở ĐÂY)

Copy `config/config.example.yaml` thành `config/config.yaml` và sửa 3 dòng đầu
(`provider`, `model_name`, `api_key_secret_name`), sau đó lưu API key cá nhân vào
Colab Secrets (icon chìa khoá bên trái) với đúng tên đã khai trong config.

> Provider `gemini` giờ dùng SDK mới `google-genai`
> (`src/llm_clients/gemini_client.py` đã cập nhật sang `genai.Client(...)` +
> `client.models.generate_content(...)`).

In [ ]:
import yaml
from google.colab import userdata

with open('config/config.yaml') as f:
    config = yaml.safe_load(f)

api_key = userdata.get(config['api_key_secret_name'])
print('Provider:', config['provider'], '| Model:', config['model_name'])

## Bước 4 — (Chỉ 1 lần) Chuẩn hoá dataset 300+300 hàm sang JSONL

In [ ]:
# Chạy 1 lần duy nhất cho cả nhóm, sau đó functions_python.jsonl / functions_java.jsonl
# nằm trong Drive dùng chung, các lần chạy sau bỏ qua bước này.
# !python data/prepare_dataset.py

## Bước 5 — Khâu 1: sinh test bằng LLM (Python + Java)

In [ ]:
import sys
sys.path.append(PROJECT_ROOT)
from src.generate_tests import run as generate_tests_run

generate_tests_run('python', config, api_key)
generate_tests_run('java', config, api_key)

## Bước 6a — Khâu 2 (Python): đo branch coverage

In [ ]:
from pathlib import Path
from src.generate_tests import load_jsonl, _tag
from src.python_coverage_runner import run_batch as run_batch_python

tag = _tag(config)
functions_py = load_jsonl(config['dataset']['python_path'])
tests_dir_py = Path(config['paths']['extracted_tests_dir'].format(tag=tag))
out_csv_py = Path(f"outputs/results_{tag}_python.csv")

run_batch_python(functions_py, tests_dir_py, out_csv_py)

## Bước 6b — Khâu 2 (Java): cài JaCoco/JUnit (1 lần) rồi đo branch coverage

In [ ]:
!java -version
!mkdir -p tools && cd tools \
  && wget -q https://repo1.maven.org/maven2/org/junit/platform/junit-platform-console-standalone/1.11.4/junit-platform-console-standalone-1.11.4.jar -O junit-platform-console-standalone.jar \
  && wget -q https://repo1.maven.org/maven2/org/jacoco/org.jacoco.agent/0.8.12/org.jacoco.agent-0.8.12-runtime.jar -O jacocoagent.jar \
  && wget -q https://repo1.maven.org/maven2/org/jacoco/org.jacoco.cli/0.8.12/org.jacoco.cli-0.8.12-nodeps.jar -O jacococli.jar \
  && ls -la
# Kiểm tra cả 3 file đều >0 byte (nếu 0 byte hoặc thiếu file -> version url sai, vào
# https://mvnrepository.com/artifact/org.junit.platform/junit-platform-console-standalone
# và https://mvnrepository.com/artifact/org.jacoco/org.jacoco.cli để lấy version mới nhất)


In [ ]:
from pathlib import Path
from src.generate_tests import load_jsonl, _tag
from src.java_coverage_runner import run_batch as run_batch_java

tag = _tag(config)
functions_java = load_jsonl(config['dataset']['java_path'])
tests_dir_java = Path(config['paths']['extracted_tests_dir'].format(tag=tag))
out_csv_java = Path(f"outputs/results_{tag}_java.csv")

run_batch_java(functions_java, tests_dir_java, out_csv_java)

## Bước 7 — Khâu 3: gộp kết quả toàn nhóm & trả lời RQ1

Chạy sau khi các thành viên đã có đủ file `results_*.csv` trong Drive dùng chung.

In [ ]:
from src.aggregate_results import aggregate, summarize_by_cc

all_df = aggregate(Path('outputs'), Path('outputs/master_results.csv'))
summary, infra_fail = summarize_by_cc(all_df)

print("=== RQ1: branch coverage theo CC bucket (chỉ tính mẫu đo được: ok/test_failed) ===")
display(summary)

print("\n=== Lỗi hạ tầng bị loại khỏi mẫu trên (theo bucket + loại lỗi) ===")
display(infra_fail)
